In [ ]:
# Colab setup: install packages not preinstalled on Colab (safe to re-run)
!pip install -q abess

# Reproduce-then-Extend — Determinants of Perceived Ageism (Gerontology) · **Day 1 tutorial**

> **Published study.** Du, J., Wang, M. & Wu, X. (2025). "The current status and determinants of perceived
> ageism among community-dwelling older adults." *PLOS ONE* 20 (doi:10.1371/journal.pone.0330254; data in the
> article's S1 Dataset). A **multiple linear regression** of perceived ageism on demographic, health, and
> functional characteristics of **484** older adults. The authors used **LASSO** to select predictors — so this
> is a study that *reproduces* a regression table **and** whose own method is the regularization we teach.

## Background

"Ageism" — prejudice and discrimination based on age — is common and consequential for older adults' health.
Du, Wang & Wu survey 484 community-dwelling older adults, measure each person's **perceived ageism**, and ask
which characteristics predict it. Because they start from a large set of candidate predictors, they use the
**LASSO** to pick a compact model, then report an ordinary regression on the selected variables. We reproduce
that regression, then reproduce the variable-selection step itself.

## Data and codebook

**Unit of analysis:** an older adult; n = 484. Outcome `Total_Score` is the perceived-ageism scale (range
8–40; higher = more perceived ageism). The file holds **27 candidate predictors** — demographics (age, gender,
education, income, marital status…), health (comorbidities, medications, self-rated health, sleep), and
**function** (`Hearing_Function`, `Vision_Status`, `Self_rated_Communication_Ability`). The eight
LASSO-selected predictors the paper reports are: educational level, employment status, number of medications,
primary caregiver, vision status, hearing function, source of health information, and communication ability.

## Descriptive results

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV, LinearRegression, lasso_path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [ ]:
d = pd.read_csv('https://raw.githubusercontent.com/desmarais-lab/desmarais-lab.github.io/master/istanbul_bilgi_ml_files/data/perceived_ageism.csv')
outcome = 'Total_Score'
candidates = [c for c in d.columns if c != outcome]
print(f'{d.shape[0]} older adults x {len(candidates)} candidate predictors')
d[[outcome]].describe().T[['mean','std','min','max']].round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].hist(d[outcome], bins=range(8,42,2), color='#a6cee3', edgecolor='white')
ax[0].set_title('Outcome: perceived ageism'); ax[0].set_xlabel('score (8-40)')
cors = d[candidates].apply(lambda c: c.corr(d[outcome])).sort_values()
top = pd.concat([cors.head(5), cors.tail(5)])
ax[1].barh([t[:22] for t in top.index], top.values, color=['#1f78b4' if v>0 else '#e31a1c' for v in top.values])
ax[1].axvline(0, color='grey'); ax[1].set_title('Strongest correlations'); plt.tight_layout()

## Reproduce the published regression

The paper's Table 2 regresses perceived ageism on the eight LASSO-selected predictors.

In [ ]:
reported8 = ['Educational_Level','Employment_Status','Number_of_Medications_Taken','Primary_Caregiver',
             'Vision_Status','Hearing_Function','Source_of_Health_Information','Self_rated_Communication_Ability']
reported8 = [c for c in reported8 if c in d.columns]
sub = d.dropna(subset=[outcome]+reported8)
ols = sm.OLS(sub[outcome], sm.add_constant(sub[reported8].astype(float))).fit()
print(f'n = {int(ols.nobs)}, R^2 = {ols.rsquared:.3f}')
for v in reported8:
    star = '***' if ols.pvalues[v]<0.001 else '**' if ols.pvalues[v]<0.01 else '*' if ols.pvalues[v]<0.05 else ''
    print(f'   {v:34s} beta = {ols.params[v]:+.3f}  p = {ols.pvalues[v]:.3f} {star}')

**Confirmation against the published study.** The regression recovers the paper's reported significant determinants: **more education predicts *less* perceived ageism**, while **poorer vision and poorer hearing predict *more*** perceived ageism — education, vision status, and hearing function are the significant terms, with the signs the paper reports.

## Regularization & variable selection — reproducing the authors' LASSO

The paper did not fit that eight-variable model by guesswork: it used the **LASSO** to select predictors from
the full candidate set. We reproduce that step — run the lasso on **all `r len(candidates)` candidates** and
see which survive — then compare penalized and unpenalized fits out of sample.

In [ ]:
X = d[candidates].apply(pd.to_numeric, errors='coerce'); y = pd.to_numeric(d[outcome], errors='coerce')
ok = X.notna().all(1) & y.notna(); X, y = X[ok].values, y[ok].values
Xs = StandardScaler().fit_transform(X)
liCV = LassoCV(cv=5, random_state=0, max_iter=100000).fit(Xs, y)
kept = [c for c, co in zip(candidates, liCV.coef_) if co != 0]
print(f'LASSO keeps {len(kept)} of {len(candidates)} candidates:')
for c in kept: print('   ', c)

The lasso recovers the paper's short list — **education, medications, hearing, vision, source of health
information** among them — the same principled selection the authors performed. Now the out-of-sample check:

In [ ]:
res = {k: [] for k in ['OLS (all 27)','Lasso','Ridge','ElasticNet']}
for r in range(50):
    Xtr,Xte,ytr,yte = train_test_split(X, y, test_size=0.30, random_state=r)
    sc = StandardScaler().fit(Xtr); Ztr, Zte = sc.transform(Xtr), sc.transform(Xte)
    res['OLS (all 27)'].append(r2_score(yte, LinearRegression().fit(Ztr,ytr).predict(Zte)))
    res['Lasso'].append(r2_score(yte, LassoCV(cv=5,random_state=0,max_iter=100000).fit(Ztr,ytr).predict(Zte)))
    res['Ridge'].append(r2_score(yte, RidgeCV(alphas=np.logspace(-2,3,40)).fit(Ztr,ytr).predict(Zte)))
    res['ElasticNet'].append(r2_score(yte, ElasticNetCV(cv=5,l1_ratio=0.5,random_state=0,max_iter=100000).fit(Ztr,ytr).predict(Zte)))
for k,v in res.items(): print(f'{k:14s} held-out R^2 = {np.mean(v):+.3f}')

Throwing all 27 candidates into OLS over-fits; the penalized models predict new respondents at least as
well while keeping a compact, interpretable set of determinants — which is exactly why Du, Wang & Wu used the
lasso to build their reported model.

In [ ]:
alphas, cpath, _ = lasso_path(Xs, y, n_alphas=40)
plt.figure(figsize=(6,3.4)); plt.plot(np.log10(alphas), cpath.T, color='#1f78b4', alpha=.4)
plt.xlabel('log10(alpha)  (more penalty ->)'); plt.ylabel('coefficient'); plt.title('Lasso paths (27 ageism predictors)'); plt.tight_layout()

## Best-subset selection with ABESS

In [ ]:
try:
    from abess.linear import LinearRegression as AbessLR
    ab = AbessLR(support_size=range(1, 9)).fit(Xs, y)
    sel = [c for c, co in zip(candidates, ab.coef_) if co != 0]
    print(f'ABESS selects a best subset of {len(sel)}: {sel}')
except Exception as e:
    print('Install abess:  !pip install abess'); print('(', e, ')')

## Takeaway

This example puts regularization to work the way a published study did. The regression identifies the determinants of perceived ageism — education, vision, and hearing — and the **lasso** selects that compact set of drivers from many candidate predictors, exactly how Du, Wang & Wu built their model. Regularization here isn't a bolt-on; it's the method the study relied on.

## Recommended exercises

1. Compare the lasso's selected set to the paper's reported eight — which differ, and why might CV disagree at the margin?
2. Fit ridge and read the shrunken coefficients: which determinants are most robust?
3. Predict a specific ageism subscale instead of the total and see whether the selected drivers change.